In [0]:
%sql
create external table ram.default.ramaaa (id int, name string) row format delimited fields terminated by '~|' lines terminated by '\n' stored as textfile location 'abfss://unitycatalog@bhavanmetadata.dfs.core.windows.net/raw/ramaa';


In [0]:
%sql
CREATE EXTERNAL TABLE ram.default.ramaaa (
  id INT,
  name STRING
)
ROW FORMAT DELIMITED 
FIELDS TERMINATED BY '~|' 
LINES TERMINATED BY '\n'
STORED AS TEXTFILE
LOCATION 'abfss://unitycatalog@bhavanmetadata.dfs.core.windows.net/raw/mutli_delimiters.txt';

In [0]:
df = spark.read.options(delimiter='|~').csv("abfss://raw@bhavanmetadata.dfs.core.windows.net/multi_delimiters.txt")
display(df)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import functions as F
schema=StructType([
StructField('customer_id', LongType(), True),
StructField('name', StringType(), True),
StructField('city', StringType(), True),
])
df = spark.read.options(delimiter='|~').schema(schema).csv("abfss://raw@bhavanmetadata.dfs.core.windows.net/mutli_delimiters.txt")

display(df)
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ram.default.customer")


In [0]:
%sql
select * from ram.default.customer

In [0]:
data = [
    (101, "Kesav", 50000, "Engineering"),
    (102, "Ravi", 52000, "Finance"),
    (103, "Ram", 51000, "Marketing"),
    (104, "Raj", 53000, "HR"),
    (105, "Sita", 54000, "Sales"),
    (106, "Meena", 55000, "Operations"),
    (107, "Anil", 56000, "Legal"),
    (108, "Sunil", 57000, "IT"),
    (109, "Priya", 58000, "Support"),
    (110, "Vijay", 59000, "R&D"),
    (111, "Asha", 60000, "Procurement"),
    (112, "Deepak", 61000, "Logistics")
]

In [0]:
df=spark.createDataFrame(data, schema=["customer_id", "name", "salary","dept"])
df.show()

In [0]:
df.groupBy('dept').agg(sum('salary')).show()

In [0]:
from pyspark.sql.functions import *

df1 = df.withColumn('status', when(col('salary') <= 50000, 'low').when(col('salary') > 50000, 'high').otherwise('na'))
df1.show()

In [0]:
df1.filter("status='high'").agg(sum('salary')).show()

In [0]:
print(df.count())

In [0]:
employee_data = [
    (201, "Arjun", 70000, "2022-01-15"),
    (202, "Neha", 72000, "2021-03-10"),
    (203, "Rohan", 71000, "2023-07-22"),
    (204, "Sneha", 73000, "2020-11-05"),
    (205, "Vikas", 74000, "2022-09-30"),
    (206, "Priya", 75000, "2021-06-18"),
    (207, "Amit", 76000, "2023-02-14"),
    (208, "Kiran", 77000, "2020-08-25"),
    (209, "Meena", 78000, "2022-12-01"),
    (210, "Sunil", 79000, "2021-04-20"),
    (211, "kesav", 80000, "2026-04-20")
]

df_employees = spark.createDataFrame(employee_data, schema=["id", "name", "salary", "hire_date"])
display(df_employees)

In [0]:
df=df_employees.withColumn('hire_date', to_date(col('hire_date'), 'yyyy-MM-dd'))
df.show()


In [0]:
df.filter("hire_date>= current_date()-90").show()

In [0]:
df.groupBy('dept').agg(sum('salary')).show()

In [0]:

df.createOrReplaceTempView("df")
spark.sql("select * from df").show()


In [0]:
%sql
select * from df

In [0]:
data = [
    (101, "Kesav", "Data Engineering", 50000, 2024),
    (101, "Kesav", "Data Engineering", 60000, 2025),
    (101, "Kesav", "Data Engineering", 70000, 2026),

    (102, "Ram", "Analytics", 45000, 2024),
    (102, "Ram", "Analytics", 52000, 2025),
    (102, "Ram", "Analytics", 58000, 2026),

    (103, "Ravi", "DevOps", 55000, 2024),
    (103, "Ravi", "DevOps", 62000, 2025),
    (103, "Ravi", "DevOps", 70000, 2026)
]

columns = ["emp_id", "emp_name", "department", "salary", "year"]

df = spark.createDataFrame(data, columns)

df.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lead


win=Window.partitionBy("emp_id").orderBy(col("year").asc())
df = df.withColumn('2nd_salary', lead('salary', 1, 0).over(win)) \
       .withColumn('2nd_year', lead('year', 1, 0).over(win)) \
       .withColumn('3rd_salary', lead('salary', 2, 0).over(win))\
        .withColumn('3rd_year', lead('year', 2, 0).over(win))
df.filter("2nd_year <> 0 and 3rd_year <> 0").show()

